# Notebook 2 — HTML registered-domain consistency
**Dataset:** PhreshPhish v1.0.1, revision `eabec4b7a66324b79cc8a0ad856d1731dc26fe1a`.

Preserve Notebook 1's 15 URL features and extract exactly 10 H candidates.
Only **internal_train**, ending **2025-07-09**, determines H08 retention.
Official test is processed mechanically; no classifiers or performance metrics appear here.

**Safety:** inspect stored HTML as text only. Never visit page URLs, fetch resources,
execute scripts, render HTML, or submit forms. The parser has no network capability.

**Input:** attach the saved Notebook 1 **FULL** output. Raw stored HTML is streamed
from the pinned Hugging Face revision. Do not upload the raw corpus manually.
**Output:** hybrid Parquets, diagnostic tables/figures and provenance manifests.
Start with `RUN_MODE = "SMOKE"`. After all 15 cells pass, change to `"FULL"`,
restart the session, and run all cells. Smoke results are not thesis results.


In [ ]:
# CELL 2: configuration, dependencies and automatic input discovery.
RUN_MODE = "SMOKE"                 # Change only this to "FULL" after smoke passes.
SMOKE_ROWS = 200
import sys, subprocess, importlib.metadata as md
PINS = {"datasets":"4.4.1", "huggingface-hub":"0.36.0",
        "pyarrow":"21.0.0", "tldextract":"5.3.0", "fsspec":"2025.10.0"}
def installed(name):
    try: return md.version(name)
    except md.PackageNotFoundError: return None
needed = [f"{p}=={v}" for p,v in PINS.items() if installed(p) != v]
needed += [p for p in ["numpy","pandas","matplotlib","scipy","xlsxwriter"]
           if installed(p) is None]
if needed:
    subprocess.check_call([sys.executable,"-m","pip","install","-q",*needed])

from pathlib import Path
from importlib import resources
from functools import lru_cache
from collections import Counter
from html.parser import HTMLParser
from urllib.parse import urlsplit, urljoin
import hashlib, json, time, re, ipaddress, gc, platform, shutil
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
import tldextract, datasets, huggingface_hub, fsspec
from datasets import load_dataset
from IPython.display import display
for package, module in [("datasets",datasets),("pyarrow",pa),
        ("tldextract",tldextract),("huggingface-hub",huggingface_hub),("fsspec",fsspec)]:
    assert module.__version__ == PINS[package], "Restart session after package installation."
assert RUN_MODE in {"SMOKE","FULL"}
DATASET_ID = "phreshphish/phreshphish"
REVISION = "eabec4b7a66324b79cc8a0ad856d1731dc26fe1a"
CUTOFF = "2025-07-09"
EXPECTED_ROWS = {"train":498255,"test":168060}
EXPECTED_HASHES = {
    "train":"e9126dccb20410ada325a923803e23679920a3d8ee135686a61ed2099f074a9f",
    "test":"0b14139e38ee61c438dcc3d49b9fbc211168651d82581850464219321073cc9f"}
U = ["url_length","hostname_length","path_length","query_length","num_dots",
     "num_hyphens","num_digits","digit_ratio","num_subdomains","path_depth",
     "num_query_params","has_ip_host","has_punycode","has_at_symbol","url_entropy"]
PSL_HASH = "b69315c085d53972724b8f2df111ffc329b0c84fe0a47d62c8c91655cc774a38"
LABEL_MAP = {"benign":0,"phish":1}
ROOT = Path("/kaggle/working/nb02") / RUN_MODE.lower()
for folder in ["features","tables","figures","manifests","logs"]:
    (ROOT/folder).mkdir(parents=True,exist_ok=True)
# An interrupted rerun must not leave an old 'ready' manifest.
for name in ["manifest_nb02.json","final_feature_set.json"]:
    (ROOT/"manifests"/name).unlink(missing_ok=True)
def sha_file(path):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for block in iter(lambda:f.read(1024*1024),b""): h.update(block)
    return h.hexdigest()
def save_json(relative, value):
    path=ROOT/relative
    tmp=path.with_suffix(path.suffix+".partial")
    tmp.write_text(json.dumps(value,indent=2,allow_nan=False),encoding="utf-8")
    tmp.replace(path)
def save_parquet(frame,path):
    tmp=path.with_suffix(".partial")
    frame.to_parquet(tmp,index=False,compression="zstd")
    tmp.replace(path)
RUNTIMES=[]
VERSIONS={p:installed(p) for p in [*PINS,"numpy","pandas","matplotlib","scipy","xlsxwriter"]}
VERSIONS["python"]=platform.python_version()
matches=[]
for p in Path("/kaggle/input").rglob("manifest_nb01.json"):
    m=json.loads(p.read_text())
    if m.get("mode")=="FULL": matches.append((p,m))
assert len(matches)==1, (
    f"Found {len(matches)} FULL NB1 manifests. Attach ONE saved NB1 FULL output; "
    "remove duplicate FULL inputs. Do not attach only a ZIP.")
NB1_MANIFEST_PATH, NB1 = matches[0]
NB1_ROOT=NB1_MANIFEST_PATH.parent.parent
ESSENTIAL=["features/url_features_train.parquet","features/url_features_test.parquet",
           "manifests/manifest_nb01.json","tables/url_feature_dictionary.csv"]
OPTIONAL=["manifests/temporal_split_preflight.json","manifests/url_feature_names.json",
          "manifests/label_mapping.json","manifests/public_suffix_list.dat"]
INPUTS={}
for rel in ESSENTIAL+OPTIONAL:
    p=NB1_ROOT/rel
    copies=list(NB1_ROOT.rglob(p.name))
    assert len(copies)<=1, f"Ambiguous copies of {p.name}: {copies}"
    if rel in ESSENTIAL: assert p.is_file(), f"Missing essential NB1 file: {p}"
    if p.is_file():
        INPUTS[rel]=p
        print(rel,"->",p)
    else: print("Optional file absent:",rel)
assert NB1.get("ready_for_nb02") is True and not NB1.get("blockers")
assert NB1.get("official_test_evaluated") is False
assert NB1.get("revision")==REVISION and NB1.get("dataset_id")==DATASET_ID
assert NB1.get("dataset_version")=="1.0.1"
assert NB1.get("rows")==EXPECTED_ROWS and NB1.get("features")==U
assert NB1.get("internal_cutoff_inclusive")==CUTOFF
assert NB1.get("psl",{}).get("sha256")==PSL_HASH
assert NB1["psl"].get("private_domains") is False
assert pd.read_csv(INPUTS["tables/url_feature_dictionary.csv"])["feature"].tolist()==U
INPUT_HASHES={rel:sha_file(p) for rel,p in INPUTS.items()}
for split in ["train","test"]:
    rel=f"features/url_features_{split}.parquet"
    assert INPUT_HASHES[rel]==EXPECTED_HASHES[split], f"NB1 {split} hash mismatch. STOP."
    recorded=NB1.get("artifacts",{}).get(rel,{})
    if recorded: assert recorded["sha256"]==INPUT_HASHES[rel]
for rel,p in INPUTS.items():
    if p.name=="url_feature_names.json": assert json.loads(p.read_text())==U
    if p.name=="label_mapping.json": assert json.loads(p.read_text())==LABEL_MAP
    if p.name=="public_suffix_list.dat": assert INPUT_HASHES[rel]==PSL_HASH
    if p.name=="temporal_split_preflight.json":
        temporal=json.loads(p.read_text())
        assert temporal["cutoff_inclusive"]==CUTOFF
        assert temporal["h08_decision_partition"]=="internal_train"
print("PASS: verified NB1 FULL inputs. Mode:",RUN_MODE,"Output:",ROOT)
print("Versions:",VERSIONS)
print("Free working disk GiB:",round(shutil.disk_usage(ROOT).free/2**30,1))


In [ ]:
# CELL 3: load compact NB1 tables, then connect to the pinned HTML stream.
URL={s:pd.read_parquet(INPUTS[f"features/url_features_{s}.parquet"])
     for s in ["train","test"]}
REQUIRED_META=["sha256","url","label","date","source_row","record_id",
               "url_parse_failed","unknown_public_suffix","registered_domain"]
for s,df in URL.items():
    assert set(REQUIRED_META+U)<=set(df.columns)
    assert len(df)==EXPECTED_ROWS[s] and df.sha256.is_unique
    assert df[["sha256","url","label","date"]].notna().all().all()
    assert df.label.isin([0,1]).all()
    assert all(pd.api.types.is_numeric_dtype(df[f]) for f in U)
    assert [c for c in df.columns if c in U]==U
    assert pd.api.types.is_datetime64_any_dtype(df.date)
    expected_labels={0:276729,1:221526} if s=="train" else {0:91260,1:76800}
    assert df.label.value_counts().to_dict()==expected_labels
    print(s,len(df),"rows;",len(df.columns),"columns")
assert not set(URL["train"].sha256)&set(URL["test"].sha256)
train=URL["train"]
assert "development_partition" in train
assert train.development_partition.value_counts().to_dict()=={
    "internal_train":397480,"internal_validation":100775}
expected_partition=np.where(train.date.dt.normalize()<=pd.Timestamp(CUTOFF),
                            "internal_train","internal_validation")
assert np.array_equal(train.development_partition.to_numpy(),expected_partition)
LOOKUP={s:df.set_index("sha256",drop=False) for s,df in URL.items()}
READ_COLUMNS=["sha256","url","label","date","html"]
def raw_stream(split):
    # One decoded HTML row at a time; plain row iteration avoids null-batch inference.
    return load_dataset(DATASET_ID,name="default",revision=REVISION,split=split,
        streaming=True,columns=READ_COLUMNS,batch_size=1)
def matched_metadata(row,split):
    key=row["sha256"]
    assert isinstance(key,str) and key in LOOKUP[split].index, "Unknown/cross-split SHA."
    m=LOOKUP[split].loc[key]
    assert row["url"]==m["url"], f"URL disagreement for SHA {key}"
    assert row["label"] in LABEL_MAP, "Unexpected raw label encoding."
    assert LABEL_MAP[row["label"]]==int(m["label"]), "Label disagreement."
    assert pd.Timestamp(row["date"])==pd.Timestamp(m["date"]), "Date disagreement."
    return m
probe=next(iter(raw_stream("train")))
assert set(probe)==set(READ_COLUMNS), "Raw projection/schema changed."
assert probe["html"] is None or isinstance(probe["html"],str)
matched_metadata(probe,"train")
print("PASS: pinned raw stream connected; projected fields:",list(probe))
print("SHA joins; every streamed URL, label and date will also be checked.")
del probe, train


## Fixed H candidates and operational rules
| ID | Feature | Reference population / definition |
|---|---|---|
| H01 | anchor_external_ratio | External / valid domain-bearing `a[href]` |
| H02 | form_external_ratio | External / valid domain-bearing `form[action]` |
| H03 | form_invalid_action_ratio | Flagged actions / all forms |
| H04 | script_external_ratio | External / valid `script[src]` |
| H05 | image_external_ratio | External / valid `img[src]` |
| H06 | iframe_external_ratio | External / valid `iframe[src]` and `frame[src]` |
| H07 | link_resource_external_ratio | External / valid `link[href]` |
| H08 | overall_external_resource_ratio | External / all valid references above, excluding H03 |
| H09 | unique_external_registered_domains | Distinct external eTLD+1 among that population |
| H10 | dominant_resource_domain_mismatch | Dominant referenced eTLD+1 differs from page eTLD+1 |

**Source map:** H01–H07 follow the hyperlink, form-action and resource-consistency
feature families in Aljofey et al. (2022), *Scientific Reports*,
[doi:10.1038/s41598-022-10841-5](https://doi.org/10.1038/s41598-022-10841-5).
The approved literature review also cites Yi et al. (2026) for external-hyperlink
ratios. H08–H10 are this study's operationalisations of those established ideas,
not claims of newly invented phishing principles.

**Frozen conventions:** relative references use `urljoin(page_url, ref)`;
`<base>` is intentionally ignored. Only HTTP(S) references with a known ICANN
registered domain enter domain denominators. IP references have no eTLD+1 and
are excluded; a syntactically valid HTTP(S) form action to an IP is not an invalid action.
Missing/empty, fragment-only, JavaScript, malformed and non-HTTP(S) form actions
are flagged by H03 and excluded from H02/H08. This is a study flag: empty and
fragment actions can have valid browser submission behaviour.
Other empty references are ignored; nonempty anchor fragments resolve to the page.
Do not extract srcset, CSS URLs, meta-refresh or JavaScript text.

Ratios with denominator zero are **0.0**; H03 with no forms and H09 with no
external domains are **0.0**. H10 with no eligible references is neutral **0.0**;
ties choose the lexicographically first ASCII eTLD+1, with a diagnostic tie flag.
Missing/blank HTML or a parser exception produces **NaN for all H values**.
An unavailable page eTLD+1 produces NaN for domain-comparison features;
H03 remains computable when the page URL itself parses. No rows are deleted.
The tolerant static parser does not certify HTML validity or browser equivalence.

The PSL is the same offline `tldextract==5.3.0` snapshot as NB1,
with private suffixes disabled. Consequently different Blogspot tenants share
`blogspot.com` under this fixed policy.

**H08:** internal_train only. Exclude if any absolute Spearman correlation with
H01–H07 is **>= 0.90**, OR H08 VIF is **> 10**. Otherwise retain, documenting
unestimable statistics. Keep H08 in descriptive data regardless. Flag >99.5%
identical observed values for review; do not automatically remove other features.
Counts, availability flags and parser diagnostics are **never predictors**.


In [ ]:
# CELL 5: offline domain parsing and static HTML event handling.
PSL_BYTES=resources.files("tldextract").joinpath(".tld_set_snapshot").read_bytes()
assert hashlib.sha256(PSL_BYTES).hexdigest()==PSL_HASH
(ROOT/"manifests/public_suffix_list.dat").write_bytes(PSL_BYTES)
DOMAIN_EXTRACTOR=tldextract.TLDExtract(suffix_list_urls=(),cache_dir=None,
                                     include_psl_private_domains=False)
@lru_cache(maxsize=50000)
def host_domain(host):
    host=host.lower().rstrip(".")
    try:
        ipaddress.ip_address(host)
        return "","ip"
    except ValueError: pass
    try:
        ascii_host=host.encode("idna").decode("ascii").lower()
        if len(ascii_host)>253 or any(not re.fullmatch(
            r"[a-z0-9_](?:[a-z0-9_-]{0,61}[a-z0-9_])?",x)
            for x in ascii_host.split(".")): return "","malformed"
        domain=DOMAIN_EXTRACTOR(ascii_host).top_domain_under_public_suffix
        return domain, "domain" if domain else "unknown_suffix"
    except UnicodeError: return "","malformed"
def web_url(text, page=False):
    if not isinstance(text,str): return None,"","malformed"
    s=text.strip()
    if not s or any(ord(c)<32 or c.isspace() for c in s):
        return None,"","malformed"
    if page:                         # Same scheme-less page policy as NB1.
        prefix=re.match(r"^([A-Za-z][A-Za-z0-9+.-]*):",s)
        explicit=bool(prefix and ("://" in s or "." not in prefix.group(1)))
        s="http:"+s if s.startswith("//") else s if explicit else "http://"+s
    try:
        p=urlsplit(s)
        if p.scheme.lower() not in {"http","https"}: return None,"","non_web"
        if not p.hostname: return None,"","malformed"
        port=p.port                  # Also reject invalid or out-of-range ports.
        domain,status=host_domain(p.hostname)
        return (None if status=="malformed" else p.geturl()),domain,status
    except (ValueError,UnicodeError): return None,"","malformed"
def resolve_reference(base,raw,is_form=False):
    if not isinstance(raw,str) or not raw.strip(): return "","empty",False
    ref=raw.strip()
    if is_form and ref.startswith("#"): return "","fragment_form",False
    if any(ord(c)<32 or c.isspace() for c in ref): return "","malformed",False
    # HTTP(S) absolute URLs require an authority; do not repair 'https:broken'.
    scheme=re.match(r"^([A-Za-z][A-Za-z0-9+.-]*):",ref)
    if scheme:
        if scheme.group(1).lower() not in {"http","https"}: return "","non_web",False
        if not re.match(r"^https?://",ref,re.I): return "","malformed",False
    try:
        absolute=urljoin(base or "",ref)
        parsed,domain,status=web_url(absolute)
        return domain,status,parsed is not None
    except (ValueError,UnicodeError): return "","malformed",False
GROUPS=["anchor","form","script","image","iframe","link_resource"]
TAG_MAP={"a":("anchor","href"),"form":("form","action"),
    "script":("script","src"),"img":("image","src"),
    "iframe":("iframe","src"),"frame":("iframe","src"),
    "link":("link_resource","href")}
class ReferenceParser(HTMLParser):
    def __init__(self,base,page_domain):
        super().__init__(convert_charrefs=True)
        self.base,self.page_domain=base,page_domain
        self.valid=Counter(); self.external=Counter(); self.elements=Counter()
        self.domains=Counter(); self.ignored=Counter()
        self.forms=0; self.invalid_forms=0
    def handle_starttag(self,tag,attrs):
        if tag not in TAG_MAP: return
        group,attribute=TAG_MAP[tag]
        self.elements[group]+=1
        # First occurrence wins if malformed markup duplicates an attribute.
        raw=next((value for name,value in attrs if name==attribute),None)
        domain,status,submit_capable=resolve_reference(self.base,raw,tag=="form")
        if tag=="form":
            self.forms+=1
            self.invalid_forms+=int(not submit_capable)
        if not domain:
            self.ignored[status]+=1
            return
        self.valid[group]+=1
        self.domains[domain]+=1
        if self.page_domain and domain!=self.page_domain: self.external[group]+=1
print("PASS: static parser and offline PSL ready; no website requests.")


In [ ]:
# CELL 6: the only master extractor; diagnostics are not predictive features.
H=["anchor_external_ratio","form_external_ratio","form_invalid_action_ratio",
   "script_external_ratio","image_external_ratio","iframe_external_ratio",
   "link_resource_external_ratio","overall_external_resource_ratio",
   "unique_external_registered_domains","dominant_resource_domain_mismatch"]
GROUP_FEATURE=dict(zip(GROUPS,[H[0],H[1],H[3],H[4],H[5],H[6]]))
DENOM={**{f:"diag_valid_"+g for g,f in GROUP_FEATURE.items()},
       H[2]:"diag_total_forms",H[7]:"diag_valid_overall"}
COUNT_COLS=(["diag_elements_"+g for g in GROUPS]+
    ["diag_valid_"+g for g in GROUPS]+["diag_total_forms","diag_invalid_forms",
    "diag_valid_overall","diag_ignored_refs","diag_ip_refs","diag_unknown_suffix_refs"])
FLAG_COLS=["diag_html_missing","diag_parser_failed","diag_page_domain_unavailable",
           "diag_dominant_tie"]
def extract_h(html,page_url,saved_domain):
    base,domain,status=web_url(page_url,page=True)
    expected="" if pd.isna(saved_domain) else str(saved_domain)
    assert domain==expected, "NB1/NB2 page-domain disagreement. Stop; do not repair silently."
    result={f:np.nan for f in H+COUNT_COLS}
    result.update({f:False for f in FLAG_COLS})
    result.update(diag_page_domain_unavailable=not bool(domain),
                  diag_parser_error="",diag_html_chars=len(html) if isinstance(html,str) else 0)
    if html is None or (isinstance(html,str) and not html.strip()):
        result["diag_html_missing"]=True
        return result
    assert isinstance(html,str), "Unexpected HTML type; source schema must be checked."
    parser=ReferenceParser(base,domain)
    try:
        # One stored page, no DOM. One feed avoids repeated copying of buffered scripts.
        parser.feed(html)
        parser.close()
    except (ValueError,AssertionError,RecursionError) as error:
        result["diag_parser_failed"]=True
        result["diag_parser_error"]=type(error).__name__
        return result
    for group in GROUPS:
        result["diag_elements_"+group]=parser.elements[group]
        result["diag_valid_"+group]=parser.valid[group]
        if domain:
            n=parser.valid[group]
            result[GROUP_FEATURE[group]]=parser.external[group]/n if n else 0.0
    total=sum(parser.valid.values())
    result.update(diag_total_forms=parser.forms,diag_invalid_forms=parser.invalid_forms,
        diag_valid_overall=total,diag_ignored_refs=sum(parser.ignored.values()),
        diag_ip_refs=parser.ignored["ip"],diag_unknown_suffix_refs=parser.ignored["unknown_suffix"])
    if base:
        result[H[2]]=parser.invalid_forms/parser.forms if parser.forms else 0.0
    if domain:
        result[H[7]]=sum(parser.external.values())/total if total else 0.0
        result[H[8]]=float(sum(d!=domain for d in parser.domains))
        if total:
            maximum=max(parser.domains.values())
            tied=sorted(d for d,n in parser.domains.items() if n==maximum)
            result["diag_dominant_tie"]=len(tied)>1
            result[H[9]]=float(tied[0]!=domain)
        else: result[H[9]]=0.0
    return result
H_SCHEMA=pa.schema([("sha256",pa.string())]+[(f,pa.float64()) for f in H+COUNT_COLS]+
    [(f,pa.bool_()) for f in FLAG_COLS]+[("diag_parser_error",pa.string()),
                                      ("diag_html_chars",pa.int64())])
print("Exactly",len(H),"H candidates. Numeric counts and flags are diagnostics only.")


In [ ]:
# CELL 7: synthetic tests; no live sites are accessed.
def sample(html,url="https://www.example.com/dir/page"):
    return extract_h(html,url,web_url(url,page=True)[1])
def close(actual,expected): assert np.isclose(actual,expected), (actual,expected)
for ref,expected in [("https://example.com/a",0),("https://other.org/a",1),
                     ("../a",0),("//other.org/a",1),("#part",0)]:
    close(sample(f'<a href="{ref}">x</a>')[H[0]],expected)
close(sample('<a href="https://cdn.example.co.uk/a">',
             "https://secure.example.co.uk")[H[0]],0)
close(sample('<a href="https://other.co.uk/a">',
             "https://secure.example.co.uk")[H[0]],1)
ip=sample('<a href="https://other.org/">',"http://192.0.2.1/")
assert np.isnan(ip[H[0]]) and ip["diag_page_domain_unavailable"]
close(sample('<a href="http://192.0.2.1/">')[H[0]],0)
assert sample('<a href="http://192.0.2.1/">')["diag_ip_refs"]==1
close(sample('<form action="/submit"></form>')[H[1]],0)
close(sample('<form action="https://other.org/go"></form>')[H[1]],1)
for action in [None,"","#x","javascript:alert(1)","mailto:a@example.com",
               "https:broken","http://[broken","https://example.com:bad"]:
    markup='<form></form>' if action is None else f'<form action="{action}"></form>'
    r=sample(markup)
    close(r[H[2]],1); close(r[H[1]],0)
close(sample('<form action="http://192.0.2.1/send"></form>')[H[2]],0)
for tag,attr,feature in [("script","src",H[3]),("img","src",H[4]),
    ("iframe","src",H[5]),("frame","src",H[5]),("link","href",H[6])]:
    close(sample(f'<{tag} {attr}="https://other.org/r"></{tag}>')[feature],1)
empty=sample('<html><body>No relevant elements</body></html>')
assert all(empty[f]==0 for f in H)
assert all(empty[d]==0 for d in DENOM.values())
assert all(np.isnan(sample(None)[f]) for f in H)
assert all(np.isnan(sample("   ")[f]) for f in H)
malformed=sample('<div><a href=https://other.org/a>unclosed<b>')
close(malformed[H[0]],1); assert not malformed["diag_parser_failed"]
many=sample('<a href="https://other.org/a">x</a>'*3+'<img src="/logo">')
close(many[H[7]],0.75); close(many[H[8]],1); close(many[H[9]],1)
tie=sample('<a href="https://other.org/"></a><a href="/a"></a>')
assert tie["diag_dominant_tie"] and tie[H[9]]==0  # example.com sorts first.
ignored=sample(''.join(f'<a href="{x}">x</a>' for x in
    ["mailto:a@other.org","tel:123","data:text/plain,x","blob:https://other.org/id"]))
assert ignored["diag_valid_overall"]==0 and ignored["diag_ignored_refs"]==4
close(sample('<base href="https://other.org/"><a href="/a">')[H[0]],0)
close(sample('<script>var x="<a href=https://other.org>";</script>')[H[7]],0)
close(sample('<a href="https://b.blogspot.com">',"https://a.blogspot.com")[H[0]],0)
close(sample('<a href="https://xn--bcher-kva.de/a">',"https://bücher.de")[H[0]],0)
assert np.isnan(sample('<a href="/">',"https://example.invalid")[H[0]])
print("PASS: all synthetic HTML/domain tests passed. Safe to begin stored-HTML extraction.")


In [ ]:
# CELL 8: small real-data smoke check and reusable bounded extraction function.
def extract_split(split,limit=None,stage="extraction"):
    started=time.perf_counter(); last_report=started
    path=ROOT/"features"/f"h_{split}_{stage}.parquet"
    tmp=path.with_suffix(".partial")
    buffer=[]; seen=set(); n=0
    source=raw_stream(split)
    if limit is not None: source=source.take(limit)
    with pq.ParquetWriter(tmp,H_SCHEMA,compression="zstd") as writer:
        for row in source:
            m=matched_metadata(row,split)
            key=row["sha256"]
            assert key not in seen, "Duplicate SHA in raw stream. STOP."
            seen.add(key)
            values=extract_h(row["html"],row["url"],m["registered_domain"])
            buffer.append({"sha256":key,**values}); n+=1
            if len(buffer)>=1000:
                writer.write_table(pa.Table.from_pylist(buffer,schema=H_SCHEMA))
                buffer.clear()
            now=time.perf_counter()
            if now-last_report>=30:
                print(split,stage,f"{n:,} pages; {now-started:.0f}s",flush=True)
                last_report=now
            del row, m, values
        if buffer: writer.write_table(pa.Table.from_pylist(buffer,schema=H_SCHEMA))
    expected=EXPECTED_ROWS[split] if limit is None else min(limit,EXPECTED_ROWS[split])
    assert n==expected, f"Incomplete {split}: {n} vs {expected}; partial file not finalised."
    if limit is None: assert seen==set(LOOKUP[split].index)
    tmp.replace(path)
    RUNTIMES.append({"stage":stage,"split":split,"rows":n,
                     "seconds":time.perf_counter()-started})
    pd.DataFrame(RUNTIMES).to_csv(ROOT/"logs/nb02_runtime.csv",index=False)
    print("PASS:",split,stage,f"{n:,} rows written.")
    return path
probe_path=extract_split("train",limit=10,stage="probe")
probe_h=pd.read_parquet(probe_path)
display(probe_h[H+FLAG_COLS].head())
print("Probe missing HTML:",int(probe_h.diag_html_missing.sum()),
      "Parser exceptions:",int(probe_h.diag_parser_failed.sum()))
print("Probe is integrity/debugging only; no feature decision is made here.")
del probe_h


In [ ]:
# CELL 9: official train includes both frozen development partitions.
limit=SMOKE_ROWS if RUN_MODE=="SMOKE" else None
H_PATHS={"train":extract_split("train",limit=limit)}
gc.collect()
print("Train extraction complete. All rows, including parse failures, are retained.")


In [ ]:
# CELL 10: test is transformed mechanically with the identical extractor.
H_PATHS["test"]=extract_split("test",limit=limit)
gc.collect()
print("Test extraction complete. No official-test performance or feature selection was computed.")


In [ ]:
# CELL 11: one-to-one joins and value preservation.
HYBRID={}; VALIDATION=[]
for split in ["train","test"]:
    h=pd.read_parquet(H_PATHS[split])
    assert h.sha256.is_unique and set(h.sha256)<=set(URL[split].sha256)
    base=URL[split].loc[URL[split].sha256.isin(h.sha256)].copy()
    joined=base.merge(h,on="sha256",how="left",validate="one_to_one",sort=False)
    expected=EXPECTED_ROWS[split] if RUN_MODE=="FULL" else SMOKE_ROWS
    assert len(joined)==len(h)==len(base)==expected
    pd.testing.assert_frame_equal(joined[base.columns].reset_index(drop=True),
                                  base.reset_index(drop=True),check_exact=True)
    for f in H[:8]: assert joined[f].dropna().between(0,1).all(), f
    assert joined[H[9]].dropna().isin([0,1]).all()
    assert (joined[H[8]].dropna()>=0).all()
    assert (joined[H[8]].dropna()%1==0).all()
    assert not np.isinf(joined[H].to_numpy()).any()
    HYBRID[split]=joined
    VALIDATION.append({"split":split,"rows":len(joined),"unique_sha":True,
                       "url_features_unchanged":True,"h_ranges_valid":True})
assert not set(HYBRID["train"].sha256)&set(HYBRID["test"].sha256)
IT=HYBRID["train"].loc[HYBRID["train"].development_partition.eq("internal_train")].copy()
assert IT.date.le(pd.Timestamp(CUTOFF)+pd.Timedelta(days=1)-pd.Timedelta(nanoseconds=1)).all()
assert len(IT)>0, "No internal_train rows in this smoke prefix; paste this message back."
if RUN_MODE=="FULL": assert len(IT)==397480
print("PASS: identities, split isolation, all 15 URL values and H ranges verified.")
display(pd.DataFrame(VALIDATION))
print("Decision/EDA rows (internal_train only):",len(IT))


In [ ]:
# CELL 12: missingness, zeros and denominator absence are separate quantities.
def percent(mask): return float(100*mask.mean()) if len(mask) else np.nan
zero_rows=[]; denominator_rows=[]; availability_rows=[]; constant_rows=[]
for label,part in [("overall",IT),("benign",IT[IT.label.eq(0)]),("phishing",IT[IT.label.eq(1)])]:
    for f in H:
        observed=part[f].dropna()
        row={"partition":"internal_train","class":label,"feature":f,"rows":len(part),
             "observed_n":len(observed),"missing_pct_all":percent(part[f].isna()),
             "zero_pct_observed":percent(observed.eq(0))}
        if f in DENOM:
            d=part[DENOM[f]]
            row["zero_with_positive_denominator_n"]=int(((part[f]==0)&(d>0)).sum())
            row["zero_with_zero_denominator_n"]=int(((part[f]==0)&(d==0)).sum())
        zero_rows.append(row)
    for f,dcol in DENOM.items():
        d=part[dcol].dropna()
        denominator_rows.append({"class":label,"feature":f,"observed_n":len(d),
            "unavailable_n":int(part[dcol].isna().sum()),"denominator_zero_pct":percent(d.eq(0)),
            "mean_denominator":float(d.mean()) if len(d) else np.nan})
    for group in GROUPS:
        elements=part["diag_elements_"+group].dropna()
        valid=part["diag_valid_"+group].dropna()
        availability_rows.append({"class":label,"group":group,"observed_n":len(elements),
            "element_present_pct":percent(elements.gt(0)),"valid_reference_present_pct":percent(valid.gt(0))})
for f in H:
    observed=IT[f].dropna()
    dominant=float(observed.value_counts(normalize=True).max()) if len(observed) else np.nan
    constant_rows.append({"feature":f,"observed_n":len(observed),
        "most_common_fraction":dominant,"review_gt_99_5_pct":bool(dominant>0.995)})
ZERO=pd.DataFrame(zero_rows); DENOM_SUM=pd.DataFrame(denominator_rows)
AVAIL=pd.DataFrame(availability_rows); CONSTANT=pd.DataFrame(constant_rows)
# Technical QC only for test: no class-conditioned EDA and no predictive decisions.
qc=[]
for split,df in HYBRID.items():
    qc.append({"split":split,"rows":len(df),"html_missing_n":int(df.diag_html_missing.sum()),
        "parser_exception_n":int(df.diag_parser_failed.sum()),
        "page_domain_unavailable_n":int(df.diag_page_domain_unavailable.sum()),
        "dominant_tie_n":int(df.diag_dominant_tie.sum()),
        "max_html_characters":int(df.diag_html_chars.max()),
        **{f+"_missing_n":int(df[f].isna().sum()) for f in H}})
QC=pd.DataFrame(qc)
TABLES={"html_zero_inflation_summary":ZERO,"html_denominator_summary":DENOM_SUM,
        "html_element_availability":AVAIL,"html_constant_feature_review":CONSTANT,
        "html_parser_failure_summary":QC}
for name,frame in TABLES.items(): frame.to_csv(ROOT/"tables"/f"{name}.csv",index=False)
display(ZERO[ZERO["class"].eq("overall")])
display(CONSTANT[CONSTANT.review_gt_99_5_pct])
print("PASS: zeros, absent denominators and missing values reported separately.")


In [ ]:
# CELL 13: internal_train only; VIF is an auxiliary linear diagnostic, not a classifier.
CORR=IT[H].corr(method="spearman",min_periods=3)
max_rho=float(CORR.loc[H[7],H[:7]].abs().max())
complete=IT[H].dropna()
varying=[f for f in H if complete[f].nunique()>1]
vif_rows=[]
if varying and len(complete)>len(varying)+1:
    x=complete[varying].to_numpy(dtype=float)
    x=(x-x.mean(axis=0))/x.std(axis=0)
    for j,f in enumerate(varying):
        y=x[:,j]; others=np.delete(x,j,axis=1)
        fitted=others@np.linalg.lstsq(others,y,rcond=None)[0] if others.shape[1] else np.zeros_like(y)
        residual_fraction=float(np.sum((y-fitted)**2)/np.sum(y**2))
        vif=np.inf if residual_fraction<=1e-12 else max(1.0,1.0/residual_fraction)
        vif_rows.append({"feature":f,"vif":vif,"status":"perfect_collinearity" if np.isinf(vif)
                         else "estimated","complete_case_n":len(complete)})
for f in H:
    if f not in [r["feature"] for r in vif_rows]:
        vif_rows.append({"feature":f,"vif":np.nan,"status":"constant_or_insufficient_complete_cases",
                         "complete_case_n":len(complete)})
VIF=pd.DataFrame(vif_rows).set_index("feature").loc[H].reset_index()
h08_vif=float(VIF.set_index("feature").loc[H[7],"vif"])
reasons=[]
if np.isfinite(max_rho) and max_rho>=0.90: reasons.append("max absolute Spearman >= 0.90")
if h08_vif>10: reasons.append("H08 VIF > 10")
REMOVE_H08=bool(reasons)
decision_reason="; ".join(reasons) if reasons else "Neither estimable threshold is triggered; retain by predeclared rule."
unavailable=[]
if not np.isfinite(max_rho): unavailable.append("Spearman maximum unavailable")
if np.isnan(h08_vif): unavailable.append("H08 VIF unavailable (constant/insufficient complete cases)")
if unavailable: decision_reason+=" "+"; ".join(unavailable)+"."
RETAINED_H=[f for f in H if not (REMOVE_H08 and f==H[7])]
def json_number(value):
    if np.isnan(value): return None
    return "Infinity" if np.isinf(value) else float(value)
H08_DECISION={"feature":H[7],"decision":"exclude" if REMOVE_H08 else "retain",
    "reason":decision_reason,"max_abs_spearman_h01_h07":json_number(max_rho),
    "vif":json_number(h08_vif),"decision_partition":"internal_train",
    "decision_rows":len(IT),"vif_complete_cases":len(complete),
    "vif_policy":"complete-case H design; constants excluded; centred auxiliary least squares",
    "provisional_smoke":RUN_MODE=="SMOKE"}
H08_SUM=pd.DataFrame([H08_DECISION])
CORR.to_csv(ROOT/"tables/h_feature_correlations_train.csv",index_label="feature")
VIF.to_csv(ROOT/"tables/vif_train.csv",index=False)
H08_SUM.to_csv(ROOT/"tables/h08_redundancy_summary.csv",index=False)
display(H08_SUM); display(VIF)
print("No other feature is removed. >99.5% flags remain documented review flags.")
if RUN_MODE=="SMOKE": print("SMOKE H08 decision is provisional; FULL recomputes it from all internal_train rows.")
del complete


In [ ]:
# CELL 14: internal-train EDA, compact Excel summaries and hybrid exports.
definitions=[
 "External valid anchors / valid domain-bearing anchors",
 "External valid form actions / valid domain-bearing form actions",
 "Flagged form actions / total forms",
 "External valid script sources / valid script sources",
 "External valid image sources / valid image sources",
 "External valid iframe/frame sources / valid frame sources",
 "External valid link resources / valid link resources",
 "External valid references / all supported valid references",
 "Distinct external registered domains among supported valid references",
 "Dominant referenced registered domain differs from page domain"]
DICTIONARY=pd.DataFrame({"id":[f"H{i:02d}" for i in range(1,11)],"feature":H,
    "definition":definitions,"retained_predictor":[f in RETAINED_H for f in H],
    "source":["Aljofey et al. (2022); approved literature feature family"]*7+
             ["Study operationalisation of established domain-consistency literature"]*3})
DICTIONARY.to_csv(ROOT/"tables/html_feature_dictionary.csv",index=False)
CLASS_SUMMARY=IT.groupby("label")[H].describe().T.reset_index()
CLASS_SUMMARY.columns=["feature","statistic",*CLASS_SUMMARY.columns[2:]]
CLASS_SUMMARY.to_csv(ROOT/"tables/html_class_summary_internal_train.csv",index=False)
PSL_POLICY={"package":"tldextract==5.3.0","sha256":PSL_HASH,
            "include_psl_private_domains":False,"suffix_list_urls":[],"offline":True}
PARSER_POLICY={"parser":"stdlib HTMLParser","network_requests_to_pages":False,
    "base_tag":"ignored; resolve against saved page URL","duplicate_attributes":"first wins",
    "domain_reference_population":"HTTP(S), known ICANN eTLD+1; IP refs excluded",
    "form_flag":"missing/empty/fragment/non-HTTP(S)/malformed; not browser validity",
    "h10_tie":"lexicographically first ASCII eTLD+1","h10_no_references":0.0,
    "missing_html_or_parser_exception":"all H NaN; retain row",
    "page_domain_unavailable":"domain H NaN; H03 available only if page URL parses",
    "malformed_html":"tolerant parser; does not certify browser-equivalent DOM"}
FINAL_SET={"dataset_id":DATASET_ID,"dataset_version":"1.0.1","revision":REVISION,
    "mode":RUN_MODE,"url_features":U,"candidate_h_features":H,
    "retained_h_features":RETAINED_H,"removed_h_features":[f for f in H if f not in RETAINED_H],
    "url_only_features":U,"combined_features":U+RETAINED_H,
    "h08_decision":H08_DECISION,"internal_cutoff_inclusive":CUTOFF,
    "decision_partition":"internal_train","psl":PSL_POLICY,"parser_policy":PARSER_POLICY,
    "zero_denominator_policy":"ratios=0.0; H03 no forms=0.0; H09 none=0; H10 no refs=0",
    "diagnostics_are_predictors":False,"official_test_evaluated":False}

plt.rcParams.update({"figure.facecolor":"white","axes.facecolor":"white",
    "font.size":10,"axes.spines.top":False,"axes.spines.right":False})
COLORS=["#0072B2","#D55E00"]
IDS=[f"H{i:02d}" for i in range(1,11)]
PREFIX="SMOKE — " if RUN_MODE=="SMOKE" else ""
def figure_save(fig,name):
    fig.tight_layout()
    fig.savefig(ROOT/"figures"/name,dpi=300,bbox_inches="tight")
    plt.show(); plt.close(fig)
fig,ax=plt.subplots(figsize=(9,4))
x=np.arange(len(GROUPS))
for i,label in enumerate(["benign","phishing"]):
    values=AVAIL[AVAIL["class"].eq(label)].set_index("group").loc[GROUPS,"element_present_pct"]
    ax.bar(x+(i-.5)*.36,values,width=.36,color=COLORS[i],label=label.title())
ax.set_xticks(x,GROUPS,rotation=20); ax.set_ylim(0,105)
ax.set_ylabel("Pages with element (%) of parsable HTML")
ax.set_title(PREFIX+"Element availability — internal_train"); ax.legend()
figure_save(fig,"F08_html_element_availability_by_class.png")

fig,ax=plt.subplots(figsize=(10,4)); x=np.arange(len(H))
for i,label in enumerate(["benign","phishing"]):
    values=ZERO[ZERO["class"].eq(label)].set_index("feature").loc[H,"zero_pct_observed"]
    ax.bar(x+(i-.5)*.36,values,width=.36,color=COLORS[i],label=label.title())
ax.set_xticks(x,IDS); ax.set_ylim(0,105); ax.set_ylabel("Zero values (%) of observed H values")
ax.set_title(PREFIX+"Zero inflation — internal_train"); ax.legend()
figure_save(fig,"F09_html_zero_inflation_by_feature.png")

fig,axes=plt.subplots(2,5,figsize=(14,6))
for j,(f,ax) in enumerate(zip(H,axes.flat)):
    data=[IT.loc[IT.label.eq(k),f].dropna().to_numpy() for k in [0,1]]
    box=ax.boxplot([v if len(v) else [np.nan] for v in data],patch_artist=True,
                   showfliers=False,whis=(5,95),medianprops={"color":"black"})
    for patch,color in zip(box["boxes"],COLORS): patch.set_facecolor(color)
    ax.set_xticks([1,2],["Benign","Phishing"],rotation=20)
    ax.set_title(IDS[j]); ax.set_ylabel("Count" if j==8 else "Value")
    if j!=8: ax.set_ylim(-.03,1.03)
    else:
        from matplotlib.ticker import MaxNLocator
        ax.set_ylim(bottom=0); ax.yaxis.set_major_locator(MaxNLocator(integer=True))
fig.suptitle(PREFIX+"H distributions — internal_train; boxes=IQR, whiskers=5th–95th percentiles",y=1.02)
figure_save(fig,"F10_h_feature_distributions_by_class.png")

fig,ax=plt.subplots(figsize=(8,7))
im=ax.imshow(CORR.to_numpy(),vmin=-1,vmax=1,cmap="RdBu_r")
ax.set_xticks(range(10),IDS); ax.set_yticks(range(10),IDS)
for i in range(10):
    for j in range(10):
        value=CORR.iloc[i,j]
        ax.text(j,i,"—" if pd.isna(value) else f"{value:.2f}",ha="center",va="center",
                fontsize=8,color="white" if pd.notna(value) and abs(value)>.65 else "black")
fig.colorbar(im,ax=ax,label="Spearman rho")
ax.set_title(PREFIX+"H correlations — internal_train (pairwise observed)")
figure_save(fig,"F11_h_feature_correlation_heatmap.png")

fig,(ax,note)=plt.subplots(1,2,figsize=(10,4),gridspec_kw={"width_ratios":[2,1]})
ax.bar(IDS[:7],CORR.loc[H[7],H[:7]].abs(),color=COLORS[0])
ax.axhline(.90,color=COLORS[1],linestyle="--",label="Removal threshold ≥ 0.90")
ax.set_ylim(0,1.05); ax.set_ylabel("Absolute Spearman rho with H08"); ax.legend(fontsize=8)
note.axis("off")
note.text(0,.8,f"H08 VIF: {h08_vif:.3g}\nVIF removal threshold: > 10\n\nDecision: {H08_DECISION['decision']}\nPartition: internal_train",
          va="top",fontsize=11)
fig.suptitle(PREFIX+"H08 redundancy decision — internal_train")
figure_save(fig,"F12_h08_redundancy_diagnostic.png")

for split,frame in HYBRID.items():
    # Preserve every H candidate in data; NB3 must select FINAL_SET feature lists.
    save_parquet(frame,ROOT/"features"/f"hybrid_features_{split}.parquet")
PACKAGE_TABLE=pd.DataFrame(list(VERSIONS.items()),columns=["package","version"])
FEATURE_TABLE=pd.DataFrame({"feature":U+H,"family":["URL"]*15+["HTML"]*10,
    "retained_predictor":[True]*15+[f in RETAINED_H for f in H]})
sheets={"HTML Feature Dictionary":DICTIONARY,"Parser Extraction QC":QC,
    "Element Availability":AVAIL,"Denominator Summary":DENOM_SUM,"Zero Inflation":ZERO,
    "Constant Review":CONSTANT,"Class H Summaries":CLASS_SUMMARY,
    "H Correlations":CORR.reset_index(names="feature"),"H08 Redundancy":H08_SUM,
    "VIF":VIF,"Runtime":pd.DataFrame(RUNTIMES),"Package Versions":PACKAGE_TABLE,
    "Final Feature Set":FEATURE_TABLE}
book=ROOT/"tables/NB02_HTML_Domain_Consistency_Diagnostics.xlsx"
with pd.ExcelWriter(book,engine="xlsxwriter",engine_kwargs={"options":{
    "strings_to_formulas":False,"strings_to_urls":False}}) as writer:
    for name,frame in sheets.items():
        frame.to_excel(writer,sheet_name=name,index=False)
        ws=writer.sheets[name]; ws.freeze_panes(1,0)
        ws.autofilter(0,0,len(frame),len(frame.columns)-1)
        ws.set_column(0,len(frame.columns)-1,23)
        ws.set_column(0,0,40)
print("PASS: 5 figures, diagnostic workbook and both hybrid Parquets saved.")


In [ ]:
# CELL 15: final provenance, artifact hashes and readiness gate. No Cell 16.
save_json("manifests/final_feature_set.json",FINAL_SET)
PACKAGE_TABLE.to_csv(ROOT/"manifests/package_versions_nb02.csv",index=False)
save_json("manifests/parser_policy.json",PARSER_POLICY)
pd.DataFrame(RUNTIMES).to_csv(ROOT/"logs/nb02_runtime.csv",index=False)
blockers=[]
for split in ["train","test"]:
    path=ROOT/"features"/f"hybrid_features_{split}.parquet"
    expected=EXPECTED_ROWS[split] if RUN_MODE=="FULL" else SMOKE_ROWS
    assert pq.ParquetFile(path).metadata.num_rows==expected
    assert set(U+H)<=set(pq.read_schema(path).names)
if RUN_MODE=="FULL":
    assert len(IT)==397480
    if IT[H].notna().sum().eq(0).any(): blockers.append("At least one H feature has no observed internal-train values.")
assert H08_DECISION["decision_partition"]=="internal_train"
assert FINAL_SET["official_test_evaluated"] is False
assert set(FINAL_SET["combined_features"])==set(U+RETAINED_H)
mandatory=["features/hybrid_features_train.parquet","features/hybrid_features_test.parquet",
    "tables/NB02_HTML_Domain_Consistency_Diagnostics.xlsx","tables/html_feature_dictionary.csv",
    "tables/html_zero_inflation_summary.csv","tables/html_denominator_summary.csv",
    "tables/html_parser_failure_summary.csv","tables/h_feature_correlations_train.csv",
    "tables/h08_redundancy_summary.csv","tables/vif_train.csv","manifests/final_feature_set.json"]
assert all((ROOT/f).is_file() for f in mandatory)
assert len(list((ROOT/"figures").glob("F*.png")))==5
artifacts={str(p.relative_to(ROOT)):{"bytes":p.stat().st_size,"sha256":sha_file(p)}
    for p in sorted(ROOT.rglob("*")) if p.is_file() and p.suffix!=".partial"
    and p.name!="manifest_nb02.json"}
MANIFEST={"notebook":"02_html_domain_consistency_features.ipynb","mode":RUN_MODE,
    "dataset_id":DATASET_ID,"dataset_version":"1.0.1","revision":REVISION,
    "nb01_manifest_reference":str(NB1_MANIFEST_PATH),
    "nb01_manifest_sha256":INPUT_HASHES["manifests/manifest_nb01.json"],
    "input_hashes":INPUT_HASHES,"rows":{s:len(df) for s,df in HYBRID.items()},
    "url_features":U,"candidate_h_features":H,"retained_h_features":RETAINED_H,
    "h08_decision":H08_DECISION,"internal_cutoff_inclusive":CUTOFF,
    "h08_decision_partition":"internal_train","psl":PSL_POLICY,"parser_policy":PARSER_POLICY,
    "packages":VERSIONS,"runtime":RUNTIMES,"validation":VALIDATION,
    "html_qc":QC.to_dict(orient="records"),"constant_review_flags":
        CONSTANT.loc[CONSTANT.review_gt_99_5_pct,"feature"].tolist(),
    "global_strict_temporal_boundary":False,"official_splits_unchanged":True,
    "official_test_evaluated":False,"artifacts":artifacts,"blockers":blockers,
    "ready_for_nb03":RUN_MODE=="FULL" and not blockers}
save_json("manifests/manifest_nb02.json",MANIFEST)
assert (ROOT/"manifests/manifest_nb02.json").is_file()
print("Rows:",MANIFEST["rows"])
print("H08:",H08_DECISION["decision"],"—",H08_DECISION["reason"])
print("Predictors:",len(U),"URL +",len(RETAINED_H),"HTML")
print("official_test_evaluated =",MANIFEST["official_test_evaluated"])
print("ready_for_nb03 =",MANIFEST["ready_for_nb03"],"| blockers =",blockers)
assert not blockers, "STOP: paste blockers and Cell 15 output back for review."
if RUN_MODE=="SMOKE":
    print("SMOKE PASS — debugging only. Change RUN_MODE to FULL, restart session, run all 15 cells.")
else:
    print("FULL PASS — save this Kaggle version WITH OUTPUTS. Notebook 2 is complete.")
print("Preserve the entire output folder:",ROOT)
print("NB3 inputs: hybrid_features_train.parquet, final_feature_set.json, manifest_nb02.json.")
print("Keep hybrid_features_test.parquet for Notebook 4; no test performance was evaluated.")
